In [1]:
import numpy as np
import pandas as pd

initial_df = pd.read_excel("data_clear.xlsx", parse_dates=["event_time"])

## Обрабатываем изначальный DataFrame, чтобы работать только с заданным промежутком времени

In [27]:
# 24.09.2020 - 28.02.2021
def set_time_intervals(start: str, end: str, df: pd.DataFrame) -> pd.DataFrame:
    start = list(map(int, start.split(".")))
    end = list(map(int, end.split(".")))
    start_ts = pd.Timestamp(day=start[0], month=start[1], year=start[2], tz="UTC")
    end_ts = pd.Timestamp(day=end[0], month=end[1], year=end[2], tz="UTC") + pd.Timedelta(days=1)
    time_mask = (start_ts <= df["event_time"]) & (end_ts > df["event_time"]) 
    return df[time_mask]


df = set_time_intervals("24.09.2020", "25.09.2020", initial_df)
df

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06+00:00,view,1996170,electronics.telephone,NaN,2775.30,1515915625519379968,LJuJVLEjPT
1,2020-09-24 11:57:26+00:00,view,139905,computers.components.cooler,zalman,1492.92,1515915625519379968,tdicluNnRY
2,2020-09-24 11:57:27+00:00,view,215454,NaN,NaN,853.47,1515915625513230080,4TMArHtXQy
3,2020-09-24 11:57:33+00:00,view,635807,computers.peripherals.printer,pantum,9901.47,1515915625519010048,aGFYrNgC08
4,2020-09-24 11:57:36+00:00,view,3658723,NaN,cameronsino,1380.69,1515915625510739968,aa4mmk0kwQ
...,...,...,...,...,...,...,...,...
6500,2020-09-25 23:52:32+00:00,view,4013167,electronics.telephone,samsung,105812.88,1515915625519770112,slzNWWR52H
6501,2020-09-25 23:52:47+00:00,view,1271372,auto.accessories.videoregister,trendvision,15846.18,1515915625519849984,d18IMWb3j9
6502,2020-09-25 23:56:01+00:00,view,1821557,electronics.telephone,sirius,1408.53,1515915625519849984,eJB1yyOSdh
6503,2020-09-25 23:57:42+00:00,view,3580539,electronics.telephone,motorola,14582.94,1515915625519770112,slzNWWR52H


## 1). Анализ пользователей, процентное соотношение всех событий к количеству сессий

In [30]:
from typing import Optional

class EventTypeAnalysis:
    def __init__(self, df: pd.DataFrame):
        self.LABEL = "Анализ совершённых событий пользователями"
        self.df: Optional[pd.DataFrame] = None
        self.total_sessions = len(df)

        self.views = df["has_view"].sum()
        self.carts = df["has_cart"].sum()
        self.purchases = df["has_purchase"].sum()

        self.view_percentage = round(self.views / self.total_sessions, 3)
        self.cart_percentage = round(self.carts / self.total_sessions, 3)
        self.purchases_percentage = round(self.purchases / self.total_sessions, 3)

    def make_analysis(self) -> pd.DataFrame:
        self.df = pd.DataFrame(
            columns=[self.LABEL],
            index=[
                "Колличество сессий",
                "Просмотры",
                "Добавлений в карзину",
                "Покупок",
                "% просмотров",
                "% добавлений в карзину",
                "% покупок"
                ])
        self.df.loc["Колличество сессий", self.LABEL] = self.total_sessions
        self.df.loc["Просмотры", self.LABEL] = self.views
        self.df.loc["Добавлений в карзину", self.LABEL] = self.carts
        self.df.loc["Покупок", self.LABEL] = self.purchases
        self.df.loc["% просмотров", self.LABEL] = self.view_percentage
        self.df.loc["% добавлений в карзину", self.LABEL] = self.cart_percentage
        self.df.loc["% покупок", self.LABEL] = self.purchases_percentage
        return self.df

event_type_df = df.groupby(["user_session"])["event_type"].apply(set).reset_index()
event_type_df["has_view"] = event_type_df["event_type"].apply(lambda x: "view" in x)
event_type_df["has_cart"] = event_type_df["event_type"].apply(lambda x: "cart" in x)
event_type_df["has_purchase"] = event_type_df["event_type"].apply(lambda x: "purchase" in x)

event_type_analysis = EventTypeAnalysis(event_type_df)
event_type_analysis.make_analysis()
# event_type_df[event_type_df["has_view"] == False]


,Анализ совершённых событий пользователями
Колличество сессий,4047
Просмотры,4031
Добавлений в карзину,281
Покупок,166
% просмотров,0.996
% добавлений в карзину,0.069
% покупок,0.041


## 2). Анализ времени между событиями, сколько пользователь тратит времени между просмотром товара и покупкой

In [ ]:
session_time_sorted_df = df.sort_values(["user_session", "event_time"])


def calc_time_diffs(session_df):
    
    first_view = session_df[session_df["event_type"] == "view"]["event_time"].min()
    purchase = session_df[session_df["event_type"] == "purchase"]["event_time"].min()
    
    result = {
        "price": np.nan,
        "total_seconds": np.nan
    }
    
    if pd.notna(first_view) and pd.notna(purchase) and purchase >= first_view:
        result["price"] = session_df[session_df["event_type"] == "purchase"]["price"].min()
        result["total_seconds"] = (purchase - first_view).total_seconds()
    return pd.Series(result)


time_stats = session_time_sorted_df.groupby('user_session').apply(calc_time_diffs).reset_index().dropna()
time_stats.sort_values("price")

C:\Users\cwdt176\AppData\Local\Temp\ipykernel_19668\2085560545.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_stats = session_time_sorted_df.groupby('user_session').apply(calculate_time_diffs).reset_index().dropna()


,user_session,price,total_seconds
796,BUVhV0L9sY,136.59,231.0
3216,mYRVldT3sN,248.82,1015.0
2754,fP8JhYaiQ4,813.45,8625.0
637,92s2sFcloI,813.45,105.0
3481,qzkLCLHKip,864.78,208.0
...,...,...,...
313,4WGp7upNB8,31089.45,1280.0
2188,XiAsdyAGCO,38115.57,3968.0
732,AJAlWhipcx,41718.24,108.0
1746,QZm9SzCcjF,41718.24,173.0


## 3). Аномалии в ценах, посмотреть бывают ли покупки одного товара по очень разной цене

In [6]:
purchases_df = df[df["event_type"] == "purchase"]
purchases_df

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
45,2020-09-24 12:04:10+00:00,purchase,1507291,computers.components.power_supply,supermicro,18928.59,1515915625519389952,xn6SHCnZtk
82,2020-09-24 12:15:06+00:00,purchase,822426,computers.peripherals.camera,logitech,10731.45,1515915625513570048,2gngxS29Ts
100,2020-09-24 12:19:01+00:00,purchase,4060928,electronics.video.tv,NaN,7762.14,1515915625518129920,3yFCkx2KKW
132,2020-09-24 12:25:18+00:00,purchase,4060928,electronics.video.tv,NaN,7762.14,1515915625518129920,3yFCkx2KKW
150,2020-09-24 12:29:49+00:00,purchase,1283197,computers.peripherals.nas,zyxel,10769.73,1515915625519350016,3jFpdbozOd
...,...,...,...,...,...,...,...,...
884905,2021-02-28 23:16:45+00:00,purchase,4154620,computers.components.videocards,msi,57126.81,1515915625596740096,h4fcX0qpOc
884913,2021-02-28 23:20:48+00:00,purchase,500058,computers.peripherals.printer,pantum,5829.00,1515915625610970112,CxMKMQDRAN
884917,2021-02-28 23:23:11+00:00,purchase,500058,computers.peripherals.printer,pantum,5829.00,1515915625610970112,CxMKMQDRAN
884925,2021-02-28 23:26:07+00:00,purchase,500058,computers.peripherals.printer,pantum,5829.00,1515915625610970112,CxMKMQDRAN


In [7]:

product_prices_df = df[df["event_type"] == "purchase"].groupby("product_id")["price"].apply(set).reset_index()
product_prices_df["price_count"] = product_prices_df["price"].apply(lambda x: len(x))
product_prices_df[product_prices_df["price_count"] > 1]
# если таблица пустая, значит все товары были куплены по одной цене

,product_id,price,price_count


## 4). Рейтинг категорий товаров по востребованности

In [21]:
purchase_rating_df = purchases_df.groupby("category_code")["product_id"].count().sort_values(ascending=False).reset_index()
purchase_rating_df.columns = ["Категория товара", "Колличество совершенных покупок"]
purchase_rating_df.to_csv("output.csv", index=False)

## 5). Анализ корзин

In [61]:

carts_df = purchases_df.groupby("user_session")["category_code"].apply(set).reset_index()
carts_df["cart_length"] = carts_df["category_code"].apply(len)
carts_df.sort_values("cart_length", ascending=False).head(10)

,user_session,category_code,cart_length
10527,QdBpqEeQGH,"{computers.components.videocards, computers.co...",9
13549,YO3XxnSFXX,"{computers.components.power_supply, computers....",8
22068,uDu86hUZL1,"{computers.components.videocards, computers.co...",8
24114,zMr79OiVmW,"{computers.components.power_supply, computers....",7
23293,xM3yZfQYhj,"{computers.components.videocards, computers.co...",7
8297,KqZdoypn7P,"{computers.components.videocards, computers.co...",7
4120,AIUQQQVmVd,"{computers.components.power_supply, computers....",7
19902,oXsQs6aJAr,"{computers.components.videocards, computers.co...",7
11324,SlNpDATCvK,"{computers.components.videocards, computers.co...",7
6045,F6ohHpTTBU,"{computers.components.power_supply, computers....",7


In [25]:
len("Анализ совершённых событий")


26